# 📈 Retail Sales Forecasting Using Python
### *Predicting Future Sales from Historical Retail Transactions*

**Author:** Junior Data Scientist Portfolio  
**Dataset:** UCI Machine Learning Repository — Online Retail Dataset (Transkasi 1 Des 2010 – 9 Des 2011)  
**Tools & Tech Stack:** Python, Pandas, NumPy, Matplotlib, Seaborn, Statsmodels, Scikit-learn, Google Colab  

---
## 🎯 Masalah Bisnis & Tujuan Analisis

### Masalah Bisnis
Manajemen operasional dan pengadaan stok (*inventory planning*) pada bisnis retail sering kali menghadapi masalah ketidakpastian permintaan (*demand fluctuation*). Membeli barang terlalu sedikit mengakibatkan kehabisan stok (*stockout* / *lost sales*), sedangkan membeli terlalu banyak menyebabkan biaya simpan membengkak dan risiko barang rusak/kedaluwarsa.

### Tujuan Proyek
1. Memahami, membersihkan, dan memvalidasi transaksi retail riil berdasarkan aturan bisnis yang dapat dipertanggungjawabkan.
2. Menganalisis tren penjualan, musiman (*seasonality*), dan anomali data melalui *Exploratory Data Analysis* (EDA).
3. Membangun model *Baseline Forecasting* sebagai patokan evaluasi standar.
4. Mengembangkan dan membandingkan model deret waktu (*Time Series Statistical*: Holt-Winters ETS) dan *Machine Learning* (Random Forest Regressor).
5. Menerjemahkan metrik evaluasi ke dalam wawasan (*insight*) serta rekomendasi perencanaan bisnis yang realistis.

---
## 📌 TAHAP 1: Setup & Memahami Dataset

Pada tahap ini, kita memuat dataset dari sumber resmi dan melakukan pemeriksaan awal (*data profiling*) untuk melihat dimensi baris/kolom, contoh transaksi, tipe data, dan persentase nilai yang hilang (*missing values*).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Konfigurasi visualisasi
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 100

# 1. Membaca dataset Excel
file_path = 'data/Online Retail.xlsx'
if not os.path.exists(file_path) and os.path.exists('Online Retail.xlsx'):
    file_path = 'Online Retail.xlsx'

print('⏳ Sedang membaca dataset Excel, mohon tunggu sebentar...')
df = pd.read_excel(file_path)
print('✅ Dataset berhasil dimuat!\n')

# 2. Menampilkan dimensi data
print(f'Jumlah Baris Awal  : {df.shape[0]:,}')
print(f'Jumlah Kolom Awal  : {df.shape[1]}')

# 3. Menampilkan 5 baris pertama data mentah
display(df.head())

# 4. Struktur data dan missing values awal
df.info()

---
## 📌 TAHAP 2: Data Cleaning (Pembersihan Data)

### Aturan Pembersihan Berdasarkan Keputusan Bisnis:
1. **Deduplikasi:** Menghapus baris duplikat identik agar tidak terjadi *double-counting* nilai transaksi.
2. **Mempertahankan `CustomerID` Kosong:** Sekitar 24.9% transaksi tidak memiliki identitas pelanggan (pembeli tamu/*guest checkout*). Karena tujuan proyek ini adalah **Forecasting Total Penjualan Toko**, uang dari pembeli tamu tetap merupakan pendapatan riil toko yang tidak boleh dihilangkan.
3. **Memfilter `UnitPrice <= 0`:** Harga $\le 0$ ditemukan pada baris catatan internal staf gudang (seperti *damaged*, *lost*, *adjustment*), bukan transaksi jual-beli riil ke pelanggan.
4. **Memfilter `Quantity <= 0`:** Faktur berawalan huruf 'C' (*Cancellation*) dan kuantitas negatif dipisahkan untuk mengukur **Gross Sales Demand** (permintaan belanja riil).
5. **Feature Engineering `TotalAmount`:** Dihitung dari formula $\text{Quantity} \times \text{UnitPrice}$.

In [ ]:
# 1. Deduplikasi
jumlah_duplikat = df.duplicated().sum()
df_dedup = df.drop_duplicates().copy()
print(f'Deduplikasi: {jumlah_duplikat:,} baris duplikat persis dibuang.')

# 2. Validasi tipe datetime pada kolom InvoiceDate
df_dedup['InvoiceDate'] = pd.to_datetime(df_dedup['InvoiceDate'])

# 3. Menyaring transaksi penjualan riil valid
df_clean = df_dedup[(df_dedup['Quantity'] > 0) & (df_dedup['UnitPrice'] > 0)].copy()
baris_dibuang = len(df) - len(df_clean)
print(f'Baris valid tersisa : {len(df_clean):,} baris')
print(f'Total baris dibuang : {baris_dibuang:,} baris ({(baris_dibuang/len(df))*100:.2f}%)')

# 4. Membuat kolom TotalAmount
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['UnitPrice']
print(f'Total Gross Revenue Toko: £{df_clean["TotalAmount"].sum():,.2f}')

# 5. Cek missing values dataset bersih
print('\nMissing Values pada df_clean:')
print(df_clean.isnull().sum())

# 6. Ringkasan statistik deskriptif
display(df_clean[['Quantity', 'UnitPrice', 'TotalAmount']].describe().round(2))

---
## 📌 TAHAP 3: Exploratory Data Analysis (EDA)

Pada tahap ini kita mengeksplorasi data untuk menjawab pertanyaan bisnis:
1. Bagaimana tren penjualan dari waktu ke waktu?
2. Bulan mana yang mencatat penjualan tertinggi dan terendah?
3. Apakah terdapat pola musiman mingguan (*day of week pattern*)?
4. Apakah terdapat anomali atau data yang terpotong?

In [ ]:
# Ekstraksi fitur waktu
df_clean['Date'] = df_clean['InvoiceDate'].dt.date
df_clean['DayOfWeek'] = df_clean['InvoiceDate'].dt.day_name()
df_clean['MonthYear'] = df_clean['InvoiceDate'].dt.to_period('M')

# Agregasi harian
daily_raw = df_clean.groupby('Date')['TotalAmount'].sum()
daily_raw.index = pd.to_datetime(daily_raw.index)

# Buat rentang kalender lengkap (mengisi hari tutup/tanpa transaksi dengan 0)
idx_full = pd.date_range(daily_raw.index.min(), daily_raw.index.max())
daily_sales = daily_raw.reindex(idx_full, fill_value=0)
daily_sales.index.name = 'Date'

# Agregasi bulanan
monthly_sales = df_clean.groupby('MonthYear')['TotalAmount'].sum()

# Agregasi hari dalam seminggu
order_days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_sales = df_clean.groupby('DayOfWeek')['TotalAmount'].sum().reindex(order_days).fillna(0)

print(f'Rentang Waktu: {daily_sales.index.min().strftime("%d %b %Y")} s.d. {daily_sales.index.max().strftime("%d %b %Y")}')
print(f'Total Hari Kalender: {len(daily_sales)} hari (Hari Aktif Transaksi: {len(daily_raw)}, Hari Tutup/Nol: {(daily_sales == 0).sum()})')

In [ ]:
# Visualisasi 1: Tren Penjualan Harian & 7-Day Moving Average
plt.figure(figsize=(14, 5))
plt.plot(daily_sales.index, daily_sales.values, color='#2b5c8f', lw=1.2, alpha=0.7, label='Penjualan Harian Riil (£)')
plt.plot(daily_sales.rolling(7).mean().index, daily_sales.rolling(7).mean().values, color='#e25822', lw=2.5, label='7-Day Moving Average (Tren)')
plt.title('Tren Penjualan Harian & 7-Day Moving Average (Des 2010 – Des 2011)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Tanggal', fontsize=11)
plt.ylabel('Total Penjualan (£)', fontsize=11)
plt.gca().yaxis.set_major_formatter('£{x:,.0f}')
plt.legend(frameon=True, loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Visualisasi 2: Penjualan Bulanan (Dengan Sorotan Bulan Parsial Des 2011)
plt.figure(figsize=(12, 5))
month_labels = [str(p) for p in monthly_sales.index]
bars = plt.bar(month_labels, monthly_sales.values, color='#2b5c8f', width=0.6)
# Warnai merah untuk bulan terakhir karena tidak lengkap
bars[-1].set_color('#d9534f')
plt.annotate('Data Parsial\n(Hanya s.d. 9 Des)', 
             xy=(len(bars)-1, monthly_sales.values[-1]),
             xytext=(len(bars)-2.2, monthly_sales.values[-1] + 180000),
             arrowprops=dict(facecolor='#d9534f', shrink=0.08, width=1.5, headwidth=7),
             fontweight='bold', color='#d9534f')
plt.title('Total Pendapatan per Bulan (Perhatikan: Desember 2011 Tidak Lengkap)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Bulan', fontsize=11)
plt.ylabel('Total Penjualan (£)', fontsize=11)
plt.gca().yaxis.set_major_formatter('£{x:,.0f}')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Visualisasi 3: Pola Penjualan Berdasarkan Hari dalam Seminggu
plt.figure(figsize=(10, 4.5))
plt.bar(dow_sales.index, dow_sales.values, color='#457b9d', width=0.55)
plt.title('Total Penjualan Berdasarkan Hari dalam Seminggu (Insight: Sabtu Toko Libur)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Hari', fontsize=11)
plt.ylabel('Total Penjualan (£)', fontsize=11)
plt.gca().yaxis.set_major_formatter('£{x:,.0f}')
plt.tight_layout()
plt.show()

for day, val in dow_sales.items():
    print(f'{day:<10}: £{val:,.2f}')

### 💡 Temuan Kritis dari EDA:
1. **Pola Musiman Mingguan Kuat:** Toko retail ini **tutup penuh pada hari Sabtu (£0)**! Penjualan tertinggi terjadi pada hari **Kamis (£2,19 Juta)** dan **Selasa (£2,17 Juta)**.
2. **Puncak Akhir Tahun (*Year-End Spike*):** Mulai September hingga November 2011 terlihat lonjakan signifikan (*Q4 seasonal peak*) menyambut perayaan akhir tahun dan Natal.
3. **Anomali Desember 2011:** Data Desember 2011 hanya mencakup 1 s.d. 9 Desember. Penjualan terlihat turun drastis di grafik bulanan murni karena datanya belum selesai (*incomplete month*), bukan karena bisnis merosot.

---
## 📌 TAHAP 4: Menentukan Target Forecasting

### Pemilihan Frekuensi: Harian (*Daily*) vs Bulanan (*Monthly*)
* **Jika Bulanan:** Dataset ini hanya memiliki **13 titik data** (13 bulan), dengan bulan terakhir terpotong. 13 titik data **terlalu sedikit** untuk memisahkan data latih/uji (*train/test*) dan tidak memungkinkan model statistik deret waktu mempelajari musiman tahunan secara valid.
* **Jika Harian:** Kita memiliki **374 hari kalender berurutan**. Data harian menyediakan sinyal yang kaya mengenai pola hari kerja, siklus mingguan (*7-day seasonality*), dan tren momentum terkini.

**Keputusan:** Target forecasting kita adalah **Total Nilai Penjualan Harian Toko (`daily_sales`)** dengan horizon pengujian 30 hari ke depan (*30-day forecast horizon*).

---
## 📌 TAHAP 5: Baseline Forecasting & Time-Based Split

### Mengapa Baseline Penting?
Sebelum membuat model Machine Learning yang kompleks, seorang Data Scientist wajib memiliki tolok ukur (*baseline*). Jika model canggih tidak mampu mengalahkan metode sederhana, maka model tersebut tidak layak diterapkan secara operasional.

### Metode Baseline yang Digunakan:
1. **Naive Forecast (Last Day):** Memprediksi penjualan hari ini persis sama dengan penjualan kemarin ($\hat{y}_t = y_{t-1}$).
2. **Seasonal Naive (7-Day):** Memprediksi penjualan hari ini sama dengan hari yang sama pada minggu lalu ($\hat{y}_t = y_{t-7}$).
3. **7-Day Moving Average:** Memprediksi penjualan berdasarkan rata-rata 7 hari terakhir.

> **Aturan Kritis:** Pembagian data wajib menggunakan **Time-Based Split** (urutan waktu), BUKAN *random split* k-fold biasa, untuk mencegah kebocoran informasi masa depan (*data leakage*).

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Time-Based Split: 30 hari terakhir sebagai Test Set
test_days = 30
train = daily_sales.iloc[:-test_days]
test = daily_sales.iloc[-test_days:]

print(f'Periode Training : {train.index.min().strftime("%Y-%m-%d")} s.d. {train.index.max().strftime("%Y-%m-%d")} ({len(train)} hari)')
print(f'Periode Testing  : {test.index.min().strftime("%Y-%m-%d")} s.d. {test.index.max().strftime("%Y-%m-%d")} ({len(test)} hari)')

# Fungsi evaluasi metrik
def hitung_metrik(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    # Hitung MAPE hanya pada hari aktif (non-zero) untuk menghindari pembagian dengan nol
    non_zero = y_true > 0
    mape = np.mean(np.abs((y_true[non_zero] - y_pred[non_zero]) / y_true[non_zero])) * 100
    return {'Model': model_name, 'MAE (£)': mae, 'RMSE (£)': rmse, 'MAPE (%)': mape}

tabel_hasil = []

# 1. Baseline: Naive (Kemarin)
pred_naive = test.shift(1)
pred_naive.iloc[0] = train.iloc[-1]
tabel_hasil.append(hitung_metrik(test, pred_naive, 'Baseline 1: Naive (Last Day)'))

# 2. Baseline: Seasonal Naive 7-Hari
pred_snaive = test.copy()
for i in range(len(test)):
    lag_date = test.index[i] - pd.Timedelta(days=7)
    pred_snaive.iloc[i] = train.loc[lag_date] if lag_date in train.index else test.loc[lag_date]
tabel_hasil.append(hitung_metrik(test, pred_snaive, 'Baseline 2: Seasonal Naive (7-Day)'))

# 3. Baseline: 7-Day Moving Average
pred_ma7 = pd.Series([train.iloc[-7:].mean()] * len(test), index=test.index)
tabel_hasil.append(hitung_metrik(test, pred_ma7, 'Baseline 3: 7-Day Moving Avg'))

display(pd.DataFrame(tabel_hasil).round(2))

---
## 📌 TAHAP 6 & 7: Membangun & Mengevaluasi Model Forecasting

Kita membandingkan dua pendekatan model:
1. **Model Statistik Time Series — Holt-Winters Exponential Smoothing (ETS):**
   Model statistik klasik yang secara eksplisit memodelkan komponen level, tren penambahan (*additive trend*), dan pola musiman berulang (*seasonal periods = 7*).
2. **Model Machine Learning — Random Forest Regressor:**
   Model ensemble non-linear berbasis pohon keputusan (*decision tree*) yang memanfaatkan fitur deret waktu (Lags 1, 2, 7, 14, rolling mean, serta fitur kalender hari dalam seminggu dan tanggal).

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.ensemble import RandomForestRegressor

# -----------------------------------------------------
# MODEL 1: Holt-Winters Exponential Smoothing (ETS)
# -----------------------------------------------------
# Menambahkan konstanta +1 untuk menghindari angka 0 stabil
hw_model = ExponentialSmoothing(
    train + 1,
    trend='add',
    seasonal='add',
    seasonal_periods=7,
    initialization_method='estimated'
).fit()

pred_hw = (hw_model.forecast(test_days) - 1).clip(lower=0)
pred_hw.index = test.index
tabel_hasil.append(hitung_metrik(test, pred_hw, 'Holt-Winters Exponential Smoothing'))

# -----------------------------------------------------
# MODEL 2: Machine Learning — Random Forest Regressor
# -----------------------------------------------------
# Feature engineering deret waktu
df_ts = pd.DataFrame({'Sales': daily_sales})
for lag in [1, 2, 7, 14]:
    df_ts[f'Lag_{lag}'] = df_ts['Sales'].shift(lag)
df_ts['Rolling_7_Mean'] = df_ts['Sales'].shift(1).rolling(7).mean()
df_ts['DayOfWeek'] = df_ts.index.dayofweek
df_ts['DayOfMonth'] = df_ts.index.day
df_ts['Month'] = df_ts.index.month

df_ts_clean = df_ts.dropna()
feature_cols = [c for c in df_ts_clean.columns if c != 'Sales']

# Split fitur train dan test
train_ml = df_ts_clean.loc[df_ts_clean.index < test.index[0]]
test_ml = df_ts_clean.loc[df_ts_clean.index >= test.index[0]]

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(train_ml[feature_cols], train_ml['Sales'])
pred_rf = pd.Series(rf.predict(test_ml[feature_cols]), index=test_ml.index)

tabel_hasil.append(hitung_metrik(test.loc[test_ml.index], pred_rf, 'Machine Learning: Random Forest'))

# Tampilkan Tabel Evaluasi Komparatif Lengkap
df_evaluasi = pd.DataFrame(tabel_hasil).sort_values(by='MAE (£)').reset_index(drop=True)
print('=' * 65)
print('🏆 TABEL PERBANDINGAN PERFORMA FORECASTING')
print('=' * 65)
display(df_evaluasi.round(2))

In [ ]:
# Visualisasi Perbandingan Prediksi vs Data Aktual Test Set
plt.figure(figsize=(14, 6))
plt.plot(train.iloc[-21:].index, train.iloc[-21:].values, color='gray', lw=1.2, ls=':', label='Data Latih (3 Minggu Terakhir)')
plt.plot(test.index, test.values, color='black', lw=2.2, marker='o', markersize=4, label='Penjualan Aktual (Test Set)')
plt.plot(test.index, pred_snaive.values, color='#457b9d', lw=1.5, ls='--', label='Seasonal Naive (7-Day)')
plt.plot(test.index, pred_hw.values, color='#e63946', lw=2, label='Holt-Winters ETS')
plt.plot(test_ml.index, pred_rf.values, color='#2a9d8f', lw=2, ls='-.', label='Random Forest Regressor')

plt.title('Perbandingan Hasil Prediksi Forecasting vs Penjualan Aktual (10 Nov – 9 Des 2011)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Tanggal', fontsize=11)
plt.ylabel('Total Penjualan (£)', fontsize=11)
plt.gca().yaxis.set_major_formatter('£{x:,.0f}')
plt.legend(frameon=True, loc='upper left')
plt.tight_layout()
plt.show()

---
## 📌 TAHAP 8: Interpretasi Hasil & Rekomendasi Bisnis

### 1. Pembahasan Performa Model
* **Baseline Naive (Last Day)** memiliki kesalahan terbesar (**MAE £29.825, MAPE 42.76%**). Hal ini logis karena jika toko ramai di hari Jumat lalu tutup di hari Sabtu, model naive akan salah menebak secara drastis.
* **Seasonal Naive 7-Hari** memangkas error hampir 50% (**MAE £15.372, MAPE 24.27%**). Ini membuktikan bahwa memahami pola musiman mingguan jauh lebih penting daripada sekadar menebak tren jangka pendek.
* **Random Forest Regressor** mencatat performa terbaik (**MAE £14.413, RMSE £29.696, MAPE 19.84%**), unggul karena mampu mengombinasikan momentum hari-hari sebelumnya (*Lag 1, 2*) dan siklus mingguan (*Lag 7, 14*).

### 2. Rekomendasi untuk Manajemen Toko
1. **Efisiensi Tenaga Kerja & Pergudangan:** Karena toko libur di hari Sabtu dan memiliki puncak pesanan tertinggi di hari Kamis dan Selasa, jadwal lembur staf gudang untuk *packing* dan pengiriman sebaiknya dialokasikan pada hari Selasa–Kamis.
2. **Penyediaan Safety Stock:** Dengan nilai kesalahan prediksi rata-rata $\approx$ **£14.400 per hari**, tim inventaris perlu menyiapkan penyangga stok (*safety buffer*) proporsional sebesar 15–20% dari estimasi permintaan harian untuk mencegah *stockout* pada produk *fast-moving*.
3. **Keterbatasan Analisis & Rekomendasi Lanjutan:**
   * Dataset hanya mencakup 1 tahun transaksi, sehingga pola musiman tahunan (efek Natal/Tahun Baru yang lebih panjang) belum bisa dipelajari secara optimal.
   * Di industri nyata, sistem forecasting dapat ditingkatkan secara signifikan dengan menambahkan variabel eksogen: jadwal diskon/promosi marketing, biaya iklan berbayar, dan data inventaris per SKU.